[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/Introduction-to-Machine-Learning-notebooks/blob/master/kmeans_basics.ipynb)

# K-means, in pictures

Run each cell with **Shift and Enter**. Five tasks, each one number to change,
and the answer written underneath.

K-means sorts points into groups without being told which point belongs where.
It places `k` centres, gives every point to the nearest centre, moves each
centre to the middle of its points, and repeats until nothing moves.

## 1. Three centres for a hundred and fifty flowers

The iris table again, with petal length and petal width in centimetres and the
kinds hidden. K-means is given the two measurements and the number of groups to
look for.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

COLOURS = ["#1e3a5f", "#b45309", "#64748b", "#236627", "#b3202c"]
TINTS = ["#dfe4ec", "#fbeddc", "#e9e9f4", "#e3efe4", "#f6e1e3"]

def lumps(ax, model, X, title):
    """The groups a k-means model found, the cuts between them, and the centres."""
    gx, gy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300),
                         np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300))
    zone = model.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    k = len(model.cluster_centers_)
    ax.contourf(gx, gy, zone, levels=np.arange(-0.5, k + 0.5, 1.0), colors=TINTS[:k])
    if k > 1:
        ax.contour(gx, gy, zone, levels=np.arange(0.5, k - 0.5, 1.0),
                   colors="#b45309", linewidths=1.0)
    for g in range(k):
        ax.scatter(X[model.labels_ == g, 0], X[model.labels_ == g, 1], s=14,
                   color=COLOURS[g], zorder=3)
    ax.scatter(model.cluster_centers_[:, 0], model.cluster_centers_[:, 1],
               marker="*", s=260, color="white", edgecolor="#1e3a5f",
               linewidth=1.2, zorder=4)
    ax.set_xlabel(labels[0])
    ax.set_ylabel(labels[1])
    ax.set_title(title)

def agree(group, truth):
    """How many points sit in a group whose commonest true label is their own."""
    return int(sum(np.bincount(truth[group == g]).max() for g in np.unique(group)))

from sklearn.datasets import load_iris
from sklearn.cluster import KMeans

iris = load_iris()
flowers = iris.data[:, [2, 3]]
kind = iris.target
labels = ["petal length (cm)", "petal width (cm)"]

k = 3
km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(flowers)

print("flowers in each group:", np.bincount(km.labels_).tolist())
for g in range(k):
    counts = np.bincount(kind[km.labels_ == g], minlength=3)
    print("group %d holds" % g,
          {str(n): int(c) for n, c in zip(iris.target_names, counts)})
print("flowers in a group of their own kind: %d of %d"
      % (agree(km.labels_, kind), len(flowers)))

fig, ax = plt.subplots(figsize=(5.6, 3.6))
lumps(ax, km, flowers, "k = %d" % k)
plt.show()

The stars are the centres and the orange lines are the cuts between the groups.
K-means never saw the kinds, and the groups it found still line up with them.

**Task 1.** Change `k = 3` to `k = 2` and run the cell again.

*Answer.* The picture has two groups, of 99 and 51 flowers. The setosa flowers still stand
alone, with one versicolor beside them, and versicolor and virginica have merged
into one group of 99. With `k = 3` the groups matched the kinds for 144 of 150
flowers and with `k = 2` for 100. K-means makes as many groups as it is asked
for, whatever the table holds.

## 2. Where the centres start

K-means begins with centres placed at random and improves them one step at a
time. Below, 300 points lie in five lumps, and the five starting centres are
five of the points, picked by `seed`. The inertia adds up the squared distance
from every point to its own centre, so a smaller inertia means tighter groups.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.metrics import pairwise_distances_argmin
import warnings

warnings.filterwarnings("ignore")

X, _ = make_blobs(n_samples=300, centers=5, cluster_std=0.6, random_state=3)

seed = 3
start = X[np.random.default_rng(seed).choice(len(X), 5, replace=False)]

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, steps, title in zip(axes, [0, 1, 30],
                            ["the start", "after one step", "settled"]):
    if steps == 0:
        centres, group = start, pairwise_distances_argmin(X, start)
    else:
        run = KMeans(n_clusters=5, init=start, n_init=1, max_iter=steps).fit(X)
        centres, group = run.cluster_centers_, run.labels_
    for g in range(5):
        ax.scatter(X[group == g, 0], X[group == g, 1], s=12, color=COLOURS[g])
    ax.scatter(centres[:, 0], centres[:, 1], marker="*", s=240, color="white",
               edgecolor="#1e3a5f", linewidth=1.2, zorder=4)
    ax.set_title(title)
plt.show()

print("points in each group:", np.bincount(group).tolist())
print("inertia: %.1f" % run.inertia_)

**Task 2.** Change `seed = 3` to `seed = 5` and run the cell again.

*Answer.* Two centres start in the same lump at the bottom left and split it in half, 30
points and 30, while one centre settles between two lumps at the top right and
takes both, 117 points. The inertia is 2070.5, more than nine times the 222.2 of
the first run. Of the seeds 0 to 11, only 2 and 3 reach 222.2. Asking for
`n_init=10`, as the cell in section 1 does, grows ten starts and keeps the one
with the lowest inertia, and on these points that reaches 222.2.

## 3. How many groups

K-means has to be told `k`. The inertia falls every time `k` grows, and it
reaches zero when every point has a group to itself, so the lowest inertia
cannot pick `k`. The place where the fall slows down can.

In [ ]:
most = 10
ks = range(1, most + 1)
inertia = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(flowers).inertia_
           for k in ks]

fig, ax = plt.subplots(figsize=(5.2, 3.0))
ax.plot(list(ks), inertia, marker="o", color="#1e3a5f")
ax.set_xlabel("groups asked for (k)")
ax.set_ylabel("inertia")
plt.show()

for i, k in enumerate(ks):
    fall = "" if i == 0 else "   fell by %.1f" % (inertia[i - 1] - inertia[i])
    print("k = %2d   inertia %6.1f%s" % (k, inertia[i], fall))

The curve drops steeply at first and then bends into a slow slide.

**Task 3.** Read the printed column and say which `k` you would use.

*Answer.* `k = 3`. Going from one group to two removes 464.5 of the inertia, going from
two to three removes 55.0, and every step after that removes 11.9 or less. The
bend in the curve is at 3, the number of kinds this table holds.

## 4. The units decide again

K-means measures distance to the centres, so a column of big numbers outweighs
a column of small ones. Below are 300 customers with two columns, age in years
and income in francs. The two groups differ in age and have the same income.

In [ ]:
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
group = rng.integers(0, 2, 300)
age = np.where(group == 0, rng.normal(32, 5, 300), rng.normal(48, 5, 300))
income = rng.normal(80000, 8000, 300)

unit = 1
X = np.c_[age, income / unit]
labels = ["age (years)", "income (francs / %d)" % unit]

raw = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X)
scaler = StandardScaler().fit(X)
scaled = KMeans(n_clusters=2, n_init=10, random_state=0).fit(scaler.transform(X))

print("income divided by %d" % unit)
print("customers in a group of their own kind, columns as they are: %d of 300"
      % agree(raw.labels_, group))
print("customers in a group of their own kind, each column scaled: %d of 300"
      % agree(scaled.labels_, group))

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6))
for ax, found, centres, title in (
        (axes[0], raw.labels_, raw.cluster_centers_, "columns as they are"),
        (axes[1], scaled.labels_,
         scaler.inverse_transform(scaled.cluster_centers_), "each column scaled")):
    for g in range(2):
        ax.scatter(X[found == g, 0], X[found == g, 1], s=12, color=COLOURS[g])
    ax.scatter(centres[:, 0], centres[:, 1], marker="*", s=260, color="white",
               edgecolor="#1e3a5f", linewidth=1.2, zorder=4)
    ax.set_xlabel(labels[0])
    ax.set_ylabel(labels[1])
    ax.set_title(title)
plt.tight_layout()
plt.show()

The left picture is k-means on the columns as they come, and the right one is
k-means after each column is rescaled to the same spread.

**Task 4.** Change `unit = 1` to `unit = 1000` and run the cell again. Income is
then divided by 1000, which counts it in thousands of francs.

*Answer.* The count for the columns as they are goes from 163 of 300 to 285 of 300, which
is what the scaled run scored all along. With income in francs the cut runs
across the picture at an income of about 80000 and ignores age, so each
kind of customer lands in both groups. Divided by 1000, income has the
narrower spread of the two columns and age decides the cut. Rescaling each
column to the same spread gives that for any table.

## 5. Round lumps

K-means gives every point to its nearest centre, so the cut between two groups
is a straight line halfway between their centres. Two arcs that wrap around
each other have no such cut.

In [ ]:
from sklearn.datasets import make_moons

shape = "arcs"
if shape == "arcs":
    X, truth = make_moons(n_samples=400, noise=0.28, random_state=0)
else:
    X, truth = make_blobs(n_samples=400, centers=2, cluster_std=1.0,
                          random_state=0)
labels = ["first measurement", "second measurement"]

km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X)

print("shape of the groups:", shape)
print("points in a group of their own kind: %d of %d"
      % (agree(km.labels_, truth), len(X)))

fig, ax = plt.subplots(figsize=(5.2, 3.6))
lumps(ax, km, X, "two groups, cut halfway between the centres")
plt.show()

**Task 5.** Change `shape = "arcs"` to `shape = "lumps"` and run the cell again.

*Answer.* The cut between the groups is a straight line halfway between the two centres.
On the arcs that line slices through both of them and the count is 305 of 400.
On two lumps it falls in the gap between them and the count is 391 of 400.

## What you can say now

K-means places `k` centres, gives every point to the nearest one and moves each
centre to the middle of its points until nothing moves. It has to be told `k`,
it settles somewhere different from different starts, it reads every column in
the units it was given, and it cuts the picture with straight lines halfway
between the centres, so the groups it finds are round lumps.